# Coding Agent Action Prediction — Full Pipeline (Colab)

This notebook runs the **current** github-repo-miner pipeline end to end:

1. Mine GitHub for repos containing SpecStory session logs
2. Extract developer<->agent turn pairs (with the corrected multi-block/tool-use parser)
3. Clean, deduplicate, apply a stratified per-repo cap, translate non-English prompts
4. Predict `action_type` (implement / debug / explain / review / tool_only / other) with a repo-held-out evaluation, comparing prompt-only vs. prompt+history representations
5. (Optional) LLM-assisted relabeling for validating the rule-based labels

**Important:** every step below calls the actual functions in `src/github_repo_miner` —
nothing is reimplemented inline in this notebook. That's deliberate: an earlier version
of this project *did* duplicate the SpecStory parsing logic inline in a notebook, it
quietly drifted out of sync with a bug fix made in the shared package, and the bug went
unnoticed for a long time. Keeping this notebook a thin orchestration layer over the
package is what prevents that class of bug from recurring.

**Before running in Colab:** the setup cell below `git clone`s this repository from
GitHub. Any local fixes need to be **committed and pushed** to that remote first, or
Colab will run against stale code.

## Step 1 — Install dependencies and get the code

In [ ]:
!pip -q install sentence-transformers umap-learn scikit-learn pandas requests matplotlib \
    xgboost deep-translator langdetect tqdm tabulate google-genai groq

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/medhab123/Prompt_Research.git"  # update if your fork differs
REPO_BRANCH = "cursor/local-prompt-analysis-and-cleaning"  # branch with the current fixes; switch to "main" once merged

if not Path("src/github_repo_miner/data_cleaning.py").exists():
    subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, "_pipeline_repo"], check=True)
    ROOT = Path("_pipeline_repo").resolve()
else:
    ROOT = Path(".").resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Using package from:", SRC)

## Step 2 — Confirm GPU is available (used for embeddings)

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("CUDA available. GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Runtime -> Change runtime type -> GPU for faster embeddings (CPU still works).")

## Step 3 — API tokens

- `GITHUB_TOKEN` (or `GH_TOKEN`) — required for mining (code search + higher rate limits).
- `GEMINI_API_KEY` or `GROQ_API_KEY` — optional, only needed for the LLM-labeling section (Step 8).

Add these as Colab secrets (key icon in the left sidebar) with the names above, or set
the environment variables directly. Values are read into `os.environ`; they are never
printed.

In [ ]:
import os

def _load_secret(name):
    if os.getenv(name):
        return
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
    except Exception:
        pass

for _name in ("GITHUB_TOKEN", "GH_TOKEN", "GEMINI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"):
    _load_secret(_name)

print("GITHUB_TOKEN set:", bool(os.getenv("GITHUB_TOKEN") or os.getenv("GH_TOKEN")))
print("GEMINI_API_KEY set:", bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")))
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## Step 4 — Mine GitHub for candidate repositories

Runs the repo-search + code-search discovery queries and writes a deduplicated
candidate CSV. This is rate-limited by GitHub (search API caps at ~30 req/min
authenticated); a large `max_results_per_query` can take several minutes.

In [ ]:
from github_repo_miner.pipeline import run_pipeline

pipeline_result = run_pipeline(
    output_dir=OUTPUT_DIR,
    max_results_per_query=1000,   # GitHub's hard per-query cap
    rate_limit_delay=5,
)
candidate_csv = pipeline_result.filtered_csv
print("Candidate repos:", len(pipeline_result.filtered))
print("Saved:", candidate_csv)

### Step 4b (alternative) — Skip mining, use an existing candidate CSV

Run this instead of Step 4 if you already have a `specstory_candidates_*.csv`.

In [ ]:
# Uncomment and set the path if skipping Step 4:
# candidate_csv = Path("outputs/specstory_candidates_YYYYMMDD_HHMM.csv")

## Step 5 — Extract developer<->agent turn pairs

For every candidate repo, discovers `.specstory/history/*.md` files and parses them into
turn pairs with the **corrected parser** (`specstory_parser.py`): consecutive
`_**Agent (...)**_` blocks belonging to one logical turn are merged instead of only
keeping the first, and `<tool-use>` blocks are summarized (`[tool: Edit]`) rather than
stripped to nothing, so a turn that's "several tool calls + a closing summary" no longer
collapses to an empty response.

Progress is checkpointed every 20 repos (`*_checkpoint.csv` in `outputs/datasets/`) — if
this cell is interrupted, rerun it; already-labeled progress up to the last checkpoint
is not automatically resumed by this call (see the local pipeline's `--extract-only`
CLI for true resume-from-checkpoint), but the checkpoint file itself is safe to inspect
or merge manually if needed.

In [ ]:
from github_repo_miner.prompt_extraction import extract_specstory_turns_from_candidates

turn_pairs_csv = extract_specstory_turns_from_candidates(
    candidate_csv,
    output_dir=OUTPUT_DIR,
    checkpoint_every=20,
)
print("Saved:", turn_pairs_csv)

## Step 6 — Clean, deduplicate, cap, translate

- Drops noise (slash commands, CI/browser log pastes, IDE-agent UI artifacts) and pure
  confirmation/continuation replies ("Ok, continue.") that carry no independent intent.
- Exact-duplicate removal.
- **Stratified per-repo cap**: repos above `max_prompts_per_repo` are downsampled, but
  stratified by `action_type` (falls back to a code/tool-use signal if `action_type`
  isn't present yet) so a repo's mix of behaviors is preserved rather than lost to a
  flat random sample. This is what keeps a handful of very chatty repos from dominating
  the corpus.
- Non-English prompts are machine-translated for downstream modeling; the original text
  is preserved in `prompt_text_original`.

In [ ]:
import pandas as pd
from github_repo_miner.data_cleaning import CleaningConfig, clean_prompts_dataframe, print_cleaning_report

raw_df = pd.read_csv(turn_pairs_csv, low_memory=False)
print("Raw turn pairs:", len(raw_df), "| repos:", raw_df["full_name"].nunique())

cleaning_config = CleaningConfig(max_prompts_per_repo=50)
cleaned_df, dropped_df, cleaning_report = clean_prompts_dataframe(raw_df, config=cleaning_config)
print_cleaning_report(cleaning_report)

per_repo = cleaned_df["full_name"].value_counts()
print(f"\nAfter cleaning: {len(cleaned_df)} rows, {cleaned_df['full_name'].nunique()} repos")
print(f"Top-5 repo share: {100 * per_repo.head(5).sum() / len(cleaned_df):.1f}%")

In [ ]:
from github_repo_miner.translate_dataset import load_translation_cache, save_translation_cache, translate_text_column

cache_path = OUTPUT_DIR / ".translation_cache.json"
translation_cache = load_translation_cache(cache_path)

translated_text, n_translated = translate_text_column(
    cleaned_df["prompt_text"].tolist(),
    translation_cache,
    preserve_code=True,
    show_progress=True,
    cache_path=cache_path,   # saved incrementally so a rerun resumes instead of restarting
)
save_translation_cache(cache_path, translation_cache)

cleaned_df["prompt_text_original"] = cleaned_df["prompt_text"]
cleaned_df["prompt_text"] = translated_text
cleaned_df["was_translated"] = cleaned_df["prompt_text_original"] != cleaned_df["prompt_text"]
print(f"Translated {n_translated} of {len(cleaned_df)} prompts")

final_dataset_csv = OUTPUT_DIR / "specstory_prompts_final.csv"
cleaned_df.to_csv(final_dataset_csv, index=False, encoding="utf-8-sig")
print("Saved:", final_dataset_csv)

## Step 7 — Predict `action_type`

Compares five representations (TF-IDF, prompt-only embedding, embedding+metadata,
embedding+history, embedding+metadata+history) across four model families
(logistic regression, random forest, MLP, XGBoost), against a majority-class baseline
and a prompt-only keyword-rule baseline. Evaluated on a **repository-held-out** split
(`GroupShuffleSplit`) — no repository appears in both train and test.

Response-side signals (`has_generated_code`, `code_block_count`, response length, ...)
are deliberately excluded from the model inputs: `action_type` is partly derived from
those same signals, so using them as features would make the task circular.

In [ ]:
from github_repo_miner.ml_research import ActionPredictionConfig, run_action_prediction

action_config = ActionPredictionConfig(
    input_csv=final_dataset_csv,
    output_dir=OUTPUT_DIR / "ml_action",
    device=DEVICE,
)
report_path = run_action_prediction(action_config)
print("Report:", report_path)

In [ ]:
print((OUTPUT_DIR / "ml_action" / "ACTION_PREDICTION_REPORT.md").read_text(encoding="utf-8"))

## Step 8 (optional) — LLM-assisted relabeling

The rule-based `action_type` label is cheap but brittle keyword matching. This asks an
LLM (Gemini or Groq — whichever key is set in Step 3) to read the prompt + response +
metadata and assign a label directly, with a confidence score and rationale, as a check
on the rule-based labels. Labels a stratified sample first (default 300 rows) rather
than the full corpus, so you can look at Cohen's kappa before committing the time/quota
to relabeling everything.

In [ ]:
from github_repo_miner.ml_research.llm_labeling import (
    ACTION_TYPES, build_human_review_sheet, cohen_kappa, label_dataframe,
)

SAMPLE_SIZE = 300
labeled_source = cleaned_df[cleaned_df["action_type"].notna()].reset_index(drop=True)

def _stratified_sample(df, total, seed=42):
    per_class = max(1, total // len(ACTION_TYPES))
    parts = [
        df[df["action_type"] == a].sample(n=min(per_class, (df["action_type"] == a).sum()), random_state=seed)
        for a in ACTION_TYPES if (df["action_type"] == a).any()
    ]
    return pd.concat(parts, ignore_index=True) if parts else df.head(0)

llm_sample = _stratified_sample(labeled_source, SAMPLE_SIZE)

llm_labels_dir = OUTPUT_DIR / "llm_labels"
llm_labeled = label_dataframe(
    llm_sample,
    cache_path=llm_labels_dir / "llm_label_cache.json",
    checkpoint_path=llm_labels_dir / "llm_labels_checkpoint.csv",
    checkpoint_every=25,
    rate_limit_delay=3.0,   # tune to your provider's free-tier rate limit
)
llm_labeled.to_csv(llm_labels_dir / "llm_labels_sample.csv", index=False, encoding="utf-8-sig")

kappa = cohen_kappa(llm_labeled["action_type"], llm_labeled["llm_action_type"])
print(f"Cohen's kappa (LLM vs. rule-based): {kappa:.3f}")
for action in ACTION_TYPES:
    mask = llm_labeled["action_type"] == action
    if mask.sum():
        agree = (llm_labeled.loc[mask, "llm_action_type"] == action).mean()
        print(f"  {action:<10} n={int(mask.sum()):<4} agreement={agree:.1%}")

review_sheet = build_human_review_sheet(llm_labeled, n_per_class=min(40, SAMPLE_SIZE // len(ACTION_TYPES)))
review_path = llm_labels_dir / "human_review_sheet.csv"
review_sheet.to_csv(review_path, index=False, encoding="utf-8-sig")
print("\nHuman review sheet:", review_path, f"({len(review_sheet)} rows — fill in 'human_label')")

## Step 9 — Download outputs

In [ ]:
import shutil

zip_path = shutil.make_archive("pipeline_outputs", "zip", OUTPUT_DIR)
print("Zipped:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — find the outputs under:", OUTPUT_DIR.resolve())